# Downstream federado pilot — CWRU + HSG18

Evalua si el ckpt FL pilot (`ssl_federated_pilot_patchtst_phm/ckpt_final.pt`) produce representaciones utiles en los 2 TT primary classification donde el SSL central confirmo transferencia: **CWRU** y **HSG18**.

**Coste estimado en A100**: ~3.5 h totales (4 corridas x ~50-65 min).

**Restricciones aplicables**:
- NO lanzar full FL.
- NO tocar `processed/` ni `processed_downstream/`.
- NO usar `ckpt_round005.pt` ni `ckpt_round010.pt` (solo `ckpt_final.pt`).
- NO ejecutar las 4 corridas en paralelo (serial, predecible).

**Decision post-corrida**:
- **GO full FL** si `macro_f1(fed_linear) > macro_f1(from_scratch)` en ambos datasets y `macro_f1(fed_full) >= 0.9 * macro_f1(central_full)`.
- **CONDITIONAL** si `linear_fed > from_scratch` pero `full_fed < 0.9 * full_central`.
- **NO-GO** si `linear_fed <= from_scratch`.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. colab_init

In [ ]:
!bash /content/drive/MyDrive/fm_fl_phmd/colab_init.sh

## 3. Pull al HEAD del bloque FL downstream pilot

Debe imprimir el commit `c70179f fl-down-pilot(preflight): 4 configs + test + runbook eval CWRU/HSG18` o superior, y `git status --porcelain` debe estar vacio.

In [ ]:
%cd /content/fm_fl_phmd
!git pull --ff-only
!git log -1 --oneline
!git status --porcelain

## 4. Verificar GPU

**Si NO es A100, para aqui**. T4/V100 no son objetivo: las 4 corridas tardarian el doble y consumirian tu cuota de Colab innecesariamente.

In [ ]:
!nvidia-smi | head -15

## 5. pytest preflight

Esperado: todos PASS, sin SKIP (en Colab Linux con torch sano).

In [ ]:
!python -m pytest \
  tests/test_downstream_federated_pilot_configs.py \
  tests/test_downstream_metrics.py \
  tests/test_downstream_pooling.py \
  tests/test_downstream_adaptive_batch.py -q

## 6. Verificar el checkpoint federado

Esperado: `model_state_dict` con muchos tensores; `config_hash = 082ca64313105f05`; `git_hash` comienza por `9b6c9fb`; `stage = pilot`.

In [ ]:
import torch
from pathlib import Path

CKPT_FL = Path(
    "/content/drive/MyDrive/fm_fl_phmd/checkpoints/"
    "ssl_federated_pilot/ssl_federated_pilot_patchtst_phm/ckpt_final.pt"
)

!ls -lh "{CKPT_FL}"

ck = torch.load(str(CKPT_FL), map_location="cpu", weights_only=False)
print("\nkeys top-level:", list(ck.keys()))
assert "model_state_dict" in ck, "ckpt FL no tiene model_state_dict"
sd = ck["model_state_dict"]
print(f"model_state_dict: {len(sd)} tensores, primer key: {next(iter(sd))}")
for opt_key in ("config_hash", "git_hash", "param_count", "stage",
                "run_name", "round", "epoch"):
    if opt_key in ck:
        print(f"{opt_key}: {ck[opt_key]}")

## 7. Preparar `_stdout/`

In [ ]:
!mkdir -p /content/drive/MyDrive/fm_fl_phmd/logs/downstream_federated_pilot/cwru/_stdout
!mkdir -p /content/drive/MyDrive/fm_fl_phmd/logs/downstream_federated_pilot/hsg18/_stdout

## 8. 4 dry-runs (sanity rapido, NO entrena)

Cada dry-run debe terminar con `config_hash`, `n_classes` detectado y `label_mapping`. Si alguno aborta por shards inexistentes, revisar `processed/<DATASET>/` antes de lanzar las corridas reales.

In [ ]:
CKPT_FL = (
    "/content/drive/MyDrive/fm_fl_phmd/checkpoints/"
    "ssl_federated_pilot/ssl_federated_pilot_patchtst_phm/ckpt_final.pt"
)

configs = [
    ("training/configs/downstream_cwru_fedavg_pilot_linear_probing.yaml",
     "linear_probing"),
    ("training/configs/downstream_cwru_fedavg_pilot_full_finetuning_lr1e-5.yaml",
     "full_finetuning"),
    ("training/configs/downstream_hsg18_fedavg_pilot_linear_probing.yaml",
     "linear_probing"),
    ("training/configs/downstream_hsg18_fedavg_pilot_full_finetuning_lr1e-5.yaml",
     "full_finetuning"),
]
for cfg, mode in configs:
    print(f"\n=== DRY-RUN {cfg} --mode {mode} ===")
    !python -m training.train_downstream_classification \
        --config "{cfg}" \
        --mode "{mode}" \
        --checkpoint "{CKPT_FL}" \
        --dry-run 2>&1 | tail -15

## 9. Corrida 1/4 — CWRU linear_probing

~50-60 min en A100.

In [ ]:
import time

CKPT_FL = (
    "/content/drive/MyDrive/fm_fl_phmd/checkpoints/"
    "ssl_federated_pilot/ssl_federated_pilot_patchtst_phm/ckpt_final.pt"
)

t0 = time.time()
!python -m training.train_downstream_classification \
  --config training/configs/downstream_cwru_fedavg_pilot_linear_probing.yaml \
  --mode linear_probing \
  --checkpoint "{CKPT_FL}" \
  2>&1 | tee /content/drive/MyDrive/fm_fl_phmd/logs/downstream_federated_pilot/cwru/_stdout/downstream_cwru_fedavg_pilot_linear_probing.stdout.log
print(f"\n[total] CWRU linear_probing elapsed = {time.time() - t0:.1f} s")

## 10. Corrida 2/4 — CWRU full_finetuning_lr1e-5

~50-65 min en A100.

In [ ]:
t0 = time.time()
!python -m training.train_downstream_classification \
  --config training/configs/downstream_cwru_fedavg_pilot_full_finetuning_lr1e-5.yaml \
  --mode full_finetuning \
  --checkpoint "{CKPT_FL}" \
  2>&1 | tee /content/drive/MyDrive/fm_fl_phmd/logs/downstream_federated_pilot/cwru/_stdout/downstream_cwru_fedavg_pilot_full_finetuning_lr1e-5.stdout.log
print(f"\n[total] CWRU full_ft_lr1e-5 elapsed = {time.time() - t0:.1f} s")

## 11. Corrida 3/4 — HSG18 linear_probing

~30-40 min en A100 (dataset menor que CWRU).

In [ ]:
t0 = time.time()
!python -m training.train_downstream_classification \
  --config training/configs/downstream_hsg18_fedavg_pilot_linear_probing.yaml \
  --mode linear_probing \
  --checkpoint "{CKPT_FL}" \
  2>&1 | tee /content/drive/MyDrive/fm_fl_phmd/logs/downstream_federated_pilot/hsg18/_stdout/downstream_hsg18_fedavg_pilot_linear_probing.stdout.log
print(f"\n[total] HSG18 linear_probing elapsed = {time.time() - t0:.1f} s")

## 12. Corrida 4/4 — HSG18 full_finetuning_lr1e-5

~30-40 min en A100.

In [ ]:
t0 = time.time()
!python -m training.train_downstream_classification \
  --config training/configs/downstream_hsg18_fedavg_pilot_full_finetuning_lr1e-5.yaml \
  --mode full_finetuning \
  --checkpoint "{CKPT_FL}" \
  2>&1 | tee /content/drive/MyDrive/fm_fl_phmd/logs/downstream_federated_pilot/hsg18/_stdout/downstream_hsg18_fedavg_pilot_full_finetuning_lr1e-5.stdout.log
print(f"\n[total] HSG18 full_ft_lr1e-5 elapsed = {time.time() - t0:.1f} s")

## 13. Summary federado y comparacion con central

Pega el output completo en el chat al terminar.

In [ ]:
import json
from pathlib import Path

fed_base = Path("/content/drive/MyDrive/fm_fl_phmd/logs/downstream_federated_pilot")
runs_fed = [
    ("CWRU",  "linear_probing",
     fed_base / "cwru"  / "downstream_cwru_fedavg_pilot_linear_probing"             / "run_info.json"),
    ("CWRU",  "full_finetuning_lr1e-5",
     fed_base / "cwru"  / "downstream_cwru_fedavg_pilot_full_finetuning_lr1e-5"     / "run_info.json"),
    ("HSG18", "linear_probing",
     fed_base / "hsg18" / "downstream_hsg18_fedavg_pilot_linear_probing"            / "run_info.json"),
    ("HSG18", "full_finetuning_lr1e-5",
     fed_base / "hsg18" / "downstream_hsg18_fedavg_pilot_full_finetuning_lr1e-5"    / "run_info.json"),
]

def _pick(d, *keys, default=None):
    for k in keys:
        if k in d and d[k] is not None:
            return d[k]
    return default

print(f"{'dataset':<8s} {'mode':<26s} {'macro_f1':>10s} {'bal_acc':>10s} {'acc':>8s} {'best_ep':>8s} {'elapsed':>10s} {'n_train':>10s} {'config_hash':>20s}")
print("-" * 130)
fed_rows = {}
for ds, mode, p in runs_fed:
    if not p.is_file():
        print(f"{ds:<8s} {mode:<26s} (NO run_info en {p})")
        continue
    ri = json.loads(p.read_text())
    tm = ri.get("test_metrics") or {}
    row = {
        "macro_f1": _pick(tm, "macro_f1", default=0.0),
        "balanced_accuracy": _pick(tm, "balanced_accuracy", default=0.0),
        "accuracy": _pick(tm, "accuracy", default=0.0),
        "best_epoch": ri.get("best_epoch", -1),
        "elapsed_seconds": ri.get("elapsed_seconds", 0.0),
        "n_trainable_params": ri.get("n_trainable_params", 0),
        "config_hash": ri.get("config_hash", ""),
        "checkpoint_loaded": ri.get("checkpoint_loaded", ""),
    }
    fed_rows[(ds, mode)] = row
    print(f"{ds:<8s} {mode:<26s} {row['macro_f1']:>10.4f} {row['balanced_accuracy']:>10.4f} {row['accuracy']:>8.4f} {row['best_epoch']:>8d} {row['elapsed_seconds']:>10.1f}s {row['n_trainable_params']:>10d} {row['config_hash']:>20s}")

print("\n=== Tabla central vs federado (macro_f1 test) ===")
print(f"{'dataset':<8s} | {'from_scratch':>13s} | {'central_linear':>15s} | {'central_full_1e-5':>18s} | {'fed_linear':>11s} | {'fed_full_1e-5':>14s}")
print("-" * 100)
central_known = {
    "CWRU":  {"from_scratch": 0.3503, "linear": 0.7046, "full_1e-5": 0.8292},
    "HSG18": {"from_scratch": 0.5693, "linear": 0.9056, "full_1e-5": 0.9504},
}
for ds in ("CWRU", "HSG18"):
    c = central_known[ds]
    fl_lin = fed_rows.get((ds, "linear_probing"),         {}).get("macro_f1")
    fl_ft  = fed_rows.get((ds, "full_finetuning_lr1e-5"), {}).get("macro_f1")
    fl_lin_str = f"{fl_lin:.4f}" if fl_lin is not None else "n/a"
    fl_ft_str  = f"{fl_ft:.4f}"  if fl_ft  is not None else "n/a"
    print(f"{ds:<8s} | {c['from_scratch']:>13.4f} | {c['linear']:>15.4f} | {c['full_1e-5']:>18.4f} | {fl_lin_str:>11s} | {fl_ft_str:>14s}")

print("\n=== Deltas (federado vs central) ===")
for ds in ("CWRU", "HSG18"):
    c = central_known[ds]
    fl_lin = fed_rows.get((ds, "linear_probing"),         {}).get("macro_f1")
    fl_ft  = fed_rows.get((ds, "full_finetuning_lr1e-5"), {}).get("macro_f1")
    if fl_lin is not None:
        d_lin_vs_fs = fl_lin - c["from_scratch"]
        d_lin_vs_central = fl_lin - c["linear"]
        print(f"{ds:<8s} linear:  fed - from_scratch = {d_lin_vs_fs:+.4f} | fed - central_linear = {d_lin_vs_central:+.4f}")
    if fl_ft is not None:
        d_ft_vs_fs = fl_ft - c["from_scratch"]
        d_ft_vs_central = fl_ft - c["full_1e-5"]
        print(f"{ds:<8s} full:    fed - from_scratch = {d_ft_vs_fs:+.4f} | fed - central_full  = {d_ft_vs_central:+.4f}")

print("\nPega el output completo en el chat para que se cierre la Fase 4.")